# Build a simple LLM application with chat models and prompt templates

In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

After reading this tutorial, you'll have a high level overview of:

- Using [language models](/docs/concepts/chat_models)

- Using [prompt templates](/docs/concepts/prompt_templates)

- Debugging and tracing your application using [LangSmith](https://docs.smith.langchain.com/)

Let's dive in!

## Setup

### Jupyter Notebook

This and other tutorials are perhaps most conveniently run in a [Jupyter notebooks](https://jupyter.org/). Going through guides in an interactive environment is a great way to better understand them. See [here](https://jupyter.org/install) for instructions on how to install.

### Installation

To install LangChain run:

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';
import CodeBlock from "@theme/CodeBlock";

<Tabs>
  <TabItem value="pip" label="Pip" default>
    <CodeBlock language="bash">pip install langchain</CodeBlock>
  </TabItem>
  <TabItem value="conda" label="Conda">
    <CodeBlock language="bash">conda install langchain -c conda-forge</CodeBlock>
  </TabItem>
</Tabs>



For more details, see our [Installation guide](/docs/how_to/installation).

### LangSmith

Many of the applications you build with LangChain will contain multiple steps with multiple invocations of LLM calls.
As these applications get more and more complex, it becomes crucial to be able to inspect what exactly is going on inside your chain or agent.
The best way to do this is with [LangSmith](https://smith.langchain.com).

After you sign up at the link above, make sure to set your environment variables to start logging traces:

```shell
export LANGCHAIN_TRACING_V2="true"
export LANGCHAIN_API_KEY="..."
```

Or, if in a notebook, you can set them with:

```python
import getpass
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass()
```

## Using Language Models

First up, let's learn how to use a language model by itself. LangChain supports many different language models that you can use interchangeably. For details on getting started with a specific model, refer to [supported integrations](/docs/integrations/chat/).

import ChatModelTabs from "@theme/ChatModelTabs";

<ChatModelTabs openaiParams={`model="gpt-4o-mini"`} />


In [ ]:
import os

In [ ]:
os.environ["OPENAI_API_KEY"] = "redacted"
os.environ['TAVILY_API_KEY']='redacted'
os.environ['LANGCHAIN_API_KEY']='redacted'

In [ ]:
# | output: false
# | echo: false

from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini")

Let's first use the model directly. [ChatModels](/docs/concepts/chat_models) are instances of LangChain [Runnables](/docs/concepts/runnables/), which means they expose a standard interface for interacting with them. To simply call the model, we can pass in a list of [messages](/docs/concepts/messages/) to the `.invoke` method.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage("Translate the following from English into Italian"),
    HumanMessage("hi!"),
]

model.invoke(messages)

:::tip

If we've enabled LangSmith, we can see that this run is logged to LangSmith, and can see the [LangSmith trace](https://smith.langchain.com/public/88baa0b2-7c1a-4d09-ba30-a47985dde2ea/r). The LangSmith trace reports [token](/docs/concepts/tokens/) usage information, latency, [standard model parameters](/docs/concepts/chat_models/#standard-parameters) (such as temperature), and other information.

:::

Note that ChatModels receive [message](/docs/concepts/messages/) objects as input and generate message objects as output. In addition to text content, message objects convey conversational [roles](/docs/concepts/messages/#role) and hold important data, such as [tool calls](/docs/concepts/tool_calling/) and token usage counts.

LangChain also supports chat model inputs via strings or [OpenAI format](/docs/concepts/messages/#openai-format). The following are equivalent:

```python
model.invoke("Hello")

model.invoke([{"role": "user", "content": "Hello"}])

model.invoke([HumanMessage("Hello")])
```

### Streaming

Because chat models are [Runnables](/docs/concepts/runnables/), they expose a standard interface that includes async and streaming modes of invocation. This allows us to stream individual tokens from a chat model:

In [ ]:
for token in model.stream(messages):
    print(token.content, end="|")

You can find more details on streaming chat model outputs in [this guide](/docs/how_to/chat_streaming/).

## Prompt Templates

Right now we are passing a list of messages directly into the language model. Where does this list of messages come from? Usually, it is constructed from a combination of user input and application logic. This application logic usually takes the raw user input and transforms it into a list of messages ready to pass to the language model. Common transformations include adding a system message or formatting a template with the user input.

[Prompt templates](/docs/concepts/prompt_templates/) are a concept in LangChain designed to assist with this transformation. They take in raw user input and return data (a prompt) that is ready to pass into a language model. 

Let's create a prompt template here. It will take in two user variables:

- `language`: The language to translate text into
- `text`: The text to translate

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "Translate the following from English into {language}"

prompt_template = ChatPromptTemplate.from_messages(
    [("system", system_template), ("user", "{text}")]
)

Note that `ChatPromptTemplate` supports multiple [message roles](/docs/concepts/messages/#role) in a single template. We format the `language` parameter into the system message, and the user `text` into a user message.

The input to this prompt template is a dictionary. We can play around with this prompt template by itself to see what it does by itself

In [ ]:
prompt = prompt_template.invoke({"language": "Italian", "text": "hi!"})

prompt

We can see that it returns a `ChatPromptValue` that consists of two messages. If we want to access the messages directly we do:

In [ ]:
prompt.to_messages()

Finally, we can invoke the chat model on the formatted prompt:

In [ ]:
response = model.invoke(prompt)
print(response.content)

:::tip
Message `content` can contain both text and [content blocks](/docs/concepts/messages/#aimessage) with additional structure. See [this guide](/docs/how_to/output_parser_string/) for more information.
:::

If we take a look at the [LangSmith trace](https://smith.langchain.com/public/3ccc2d5e-2869-467b-95d6-33a577df99a2/r), we can see exactly what prompt the chat model receives, along with [token](/docs/concepts/tokens/) usage information, latency, [standard model parameters](/docs/concepts/chat_models/#standard-parameters) (such as temperature), and other information.

## Conclusion

That's it! In this tutorial you've learned how to create your first simple LLM application. You've learned how to work with language models, how to create a prompt template, and how to get great observability into applications you create with LangSmith.

This just scratches the surface of what you will want to learn to become a proficient AI Engineer. Luckily - we've got a lot of other resources!

For further reading on the core concepts of LangChain, we've got detailed [Conceptual Guides](/docs/concepts).

If you have more specific questions on these concepts, check out the following sections of the how-to guides:

- [Chat models](/docs/how_to/#chat-models)
- [Prompt templates](/docs/how_to/#prompt-templates)

And the LangSmith docs:

- [LangSmith](https://docs.smith.langchain.com)

### SD Response: 

The `Lab 1 Instructions` asks us to translate Hello World to Spanish, so I am running the below as well:

In [ ]:
prompt = prompt_template.invoke({"language": "Spanish", "text": "Hello World"})

prompt

In [ ]:
prompt.to_messages()

In [ ]:
response = model.invoke(prompt)
print(response.content)

# Module 1 Class Discussion

In [ ]:
prompt = """I want to develop an AI Agent for 'Cloud Computing and E-Commerce'. \
I want your help in deciding which software framework to use. \
I have the following options: LangChain, LangGraph, AutoGen, CrewAI, and Baby AGI. \
Compare/contrast them in a table format for the output."""

response = model.invoke([HumanMessage(prompt)])
print(response.content)

Here's a comparative analysis of the mentioned software frameworks for developing an AI agent focused on Cloud Computing and E-Commerce. The comparison looks at various attributes like ease of use, scalability, community support, and unique features.

| Framework     | Ease of Use | Scalability | Community Support | Key Features                                            | Ideal Use Case                                       |
|---------------|-------------|-------------|-------------------|--------------------------------------------------------|-----------------------------------------------------|
| **LangChain** | Moderate    | High        | Strong            | Language model integration, memory management, tools and integrations | Versatile for various NLP tasks, strong in chaining language models |
| **LangGraph** | Moderate    | High        | Emerging          | Graph-based approach for structuring knowledge, focuses on relationships between entities | Use cases that require understanding complex relationships in data |
| **AutoGen**   | Easy        | Moderate    | Growing           | Automated agent generation, user-friendly interface, allows for rapid prototyping | Quickly developing AI agents with minimal configuration |
| **CrewAI**    | Easy        | High        | Medium            | Collaboration features, multi-agent interaction, effective task delegation | Suitable for team-oriented projects and collaborative AI tasks |
| **Baby AGI**  | Moderate    | High        | Limited           | Simulates basic AGI concepts, designed for experimentation with self-improving agents | Research and experimentation into general AI behaviors |

### Summary of Analysis:
1. **LangChain**: Best for projects requiring extensive NLP capabilities due to its support for various models and integration options, but it has a moderate learning curve.
  
2. **LangGraph**: Appeals to projects where relationships and data structure are crucial, though it's relatively new and may have less community support.

3. **AutoGen**: Suited for rapid development of AI agents with a simpler interface, making it a good choice for less complex e-commerce solutions but might lack in-depth features for cloud computing.

4. **CrewAI**: Well-suited for collaborative projects, particularly when multiple agents need to work together, making it excellent for team-based e-commerce solutions.

5. **Baby AGI**: Primarily for research-driven projects focusing on AGI principles; not as practical for straightforward applications in cloud computing and e-commerce but good for prototyping advanced concepts.

### Conclusion:
For a cloud computing and e-commerce AI agent, **LangChain** or **CrewAI** might be the most appropriate choices based on their scalability and robust features. If you prioritize ease of use and rapid prototyping, **AutoGen** may be the best fit. Consider your specific needs in functionality, community support, and ease of development to make the best choice.

Step 3: Evaluation of Peer’s Post

    Pick one of your classmates’ prompts, and after you copy it to your clipboard, enter it into ChatGPT (gpt-4o-mini). Compare and contrast the output you get to your classmate’s output. Give a rating to each response (Rating range 1-5 stars). Which response (yours or your classmate’s) do you rate as a better response from ChatGPT and why? 



In [ ]:
# Defining my response as a variable (fixing to avoid variability)
my_response = """Here's a comparative analysis of the mentioned software frameworks for developing an AI agent focused on Cloud Computing and E-Commerce. The comparison looks at various attributes like ease of use, scalability, community support, and unique features.

| Framework     | Ease of Use | Scalability | Community Support | Key Features                                            | Ideal Use Case                                       |
|---------------|-------------|-------------|-------------------|--------------------------------------------------------|-----------------------------------------------------|
| **LangChain** | Moderate    | High        | Strong            | Language model integration, memory management, tools and integrations | Versatile for various NLP tasks, strong in chaining language models |
| **LangGraph** | Moderate    | High        | Emerging          | Graph-based approach for structuring knowledge, focuses on relationships between entities | Use cases that require understanding complex relationships in data |
| **AutoGen**   | Easy        | Moderate    | Growing           | Automated agent generation, user-friendly interface, allows for rapid prototyping | Quickly developing AI agents with minimal configuration |
| **CrewAI**    | Easy        | High        | Medium            | Collaboration features, multi-agent interaction, effective task delegation | Suitable for team-oriented projects and collaborative AI tasks |
| **Baby AGI**  | Moderate    | High        | Limited           | Simulates basic AGI concepts, designed for experimentation with self-improving agents | Research and experimentation into general AI behaviors |

### Summary of Analysis:
1. **LangChain**: Best for projects requiring extensive NLP capabilities due to its support for various models and integration options, but it has a moderate learning curve.
  
2. **LangGraph**: Appeals to projects where relationships and data structure are crucial, though it's relatively new and may have less community support.

3. **AutoGen**: Suited for rapid development of AI agents with a simpler interface, making it a good choice for less complex e-commerce solutions but might lack in-depth features for cloud computing.

4. **CrewAI**: Well-suited for collaborative projects, particularly when multiple agents need to work together, making it excellent for team-based e-commerce solutions.

5. **Baby AGI**: Primarily for research-driven projects focusing on AGI principles; not as practical for straightforward applications in cloud computing and e-commerce but good for prototyping advanced concepts.

### Conclusion:
For a cloud computing and e-commerce AI agent, **LangChain** or **CrewAI** might be the most appropriate choices based on their scalability and robust features. If you prioritize ease of use and rapid prototyping, **AutoGen** may be the best fit. Consider your specific needs in functionality, community support, and ease of development to make the best choice."""

I picked Scott Forster's post and asked GPT-4o-mini to compare it to my post. Below is Scott Forster's post for reference:

In [ ]:
# Defining my peer's response as a variable
peer_response = """Comparison Table
Criteria 	LangChain 	LangGraph 	AutoGen 	CrewAI 	BabyAGI
Primary Purpose 	LLM application framework 	Stateful agent orchestration framework built on LangChain 	Multi-agent conversations and collaboration 	Role-based multi-agent teams 	Autonomous task creation and execution
Maturity 	Very mature 	Mature and rapidly growing 	Mature 	Growing rapidly 	Mostly experimental
Production Readiness 	High 	Very High 	Medium-High 	Medium 	Low
Learning Curve 	Moderate 	Moderate-High 	Moderate 	Easy-Moderate 	Easy
Multi-Agent Support 	Basic 	Advanced 	Excellent 	Excellent 	Limited
Workflow Control 	Moderate 	Excellent 	Good 	Moderate 	Limited
State Management 	Basic 	Excellent 	Moderate 	Moderate 	Minimal
Human-in-the-Loop Support 	Good 	Excellent 	Good 	Moderate 	Poor
Observability & Debugging 	Good 	Excellent 	Moderate 	Moderate 	Limited
Scalability 	High 	Very High 	Medium-High 	Medium 	Low
Deterministic Workflows 	Moderate 	Excellent 	Moderate 	Moderate 	Poor
Autonomous Agent Behavior 	Moderate 	High 	High 	High 	Very High
Enterprise Adoption 	High 	Increasing rapidly 	Growing 	Growing 	Low
Tool Integration 	Excellent 	Excellent 	Good 	Good 	Limited
Memory Support 	Good 	Excellent 	Good 	Good 	Basic
Best Use Cases 	Chatbots, RAG apps, tool calling 	Enterprise agents, complex business workflows 	Collaborative AI teams 	Business process automation 	Research/experimentation
Recommended for CPG Enterprise? 	Yes 	Strong Yes 	Sometimes 	Sometimes 	No
Architectural Differences
LangChain

Think of LangChain as the "toolbox."

Strengths

    Connects LLMs to:
        Databases
        APIs
        Vector stores
        Documents
    Strong ecosystem
    Large community

Weaknesses

    Complex workflows can become difficult to manage
    Agent behavior can be less predictable

CPG Example

    Product recommendation chatbot
    Sales dashboard Q&A
    Retailer support assistant

How They Compare for Common CPG Use Cases
Use Case 	Best Choice
Sales analytics agent 	LangGraph
Trade promotion optimization 	LangGraph
Demand forecasting workflow 	LangGraph
Retail account management assistant 	LangGraph
Product innovation brainstorming 	AutoGen or CrewAI
Consumer insights analysis 	AutoGen
Executive reporting agents 	CrewAI
RAG over product/specification documents 	LangChain or LangGraph
Supply chain exception management 	LangGraph
Multi-department AI team simulation 	AutoGen"""

![image.png](attachment:image.png)

![image.png](attachment:image.png)

In [ ]:
# Defining my peer's prompt
peer_prompt = """I want to develop an AI Agent for a consumer package food company example. I want your help in deciding which software framework to use. I have the following options: LangChain, LangGraph, AutoGen, CrewAI, and Baby AGI. Compare/contrast them in a table format for the output."""

In [ ]:
# Generating a response to the peer prompt
response_to_peer = model.invoke([HumanMessage(peer_prompt)])
print(response_to_peer.content)

Here's a comparison table for the software frameworks you're considering for developing an AI agent for a consumer packaged food company:

| Feature       | LangChain                             | LangGraph                             | AutoGen                              | CrewAI                                | Baby AGI                             |
|---------------|--------------------------------------|--------------------------------------|--------------------------------------|--------------------------------------|-------------------------------------|
| **Description** | A framework for building applications with large language models. It enables developers to connect different LLMs, APIs, and external data sources. | Focuses on graph-based reasoning and allows manipulation of graph structures for AI tasks. | An automated framework that assists in generating AI agents through interactivity and self-optimization. | A collaborative framework for creating AI agents that work together and utilize shared knowledge. | A modular framework designed for developing AI agents that learn and adapt over time through reinforcement learning methods. |
| **Ease of Use** | Moderate – requires understanding of LLMs | Moderate - requires a good grasp of graph theory | Relatively easy – focuses on automated generation | User-friendly interface for collaboration | Moderate to advanced – requires understanding of reinforcement learning |
| **Customization** | Highly customizable with support for various plugins | Customizable graph structures but less extensible for other tasks | Some customization options, primarily focused on automated workflows | High customization potential, especially in cooperative tasks | Customization through modular components but may require deeper ML expertise |
| **Collaboration Features** | Limited collaboration features, primarily individual agent development | Limited focus on collaborative tasks | Moderate collaboration features but mainly for automation | Strong collaboration features, emphasizing group tasks | Limited collaboration focus |
| **Performance with Large Data** | Optimized for large datasets and integrates well with various data sources | Works well with graph data but may struggle with unstructured data | Performance may vary based on generated agent tasks | Good performance with complex data integrations | Designed for continuous learning, but heavier models may impact performance |
| **Community Support** | Strong community with extensive documentation and examples | Growing community, but less mature than LangChain | Emerging community, but limited resources and tutorials | Moderate community support with some resources | Smaller community with more experimental frameworks |
| **Ideal Use Cases** | Versatile applications like chatbots, data analysis, and content generation | Specific applications requiring graph-based reasoning | Rapid prototyping of AI agents | Collaborative tasks such as team support systems | Continuous learning and task adaptation scenarios |
| **Integration Capabilities** | Extensive integrations with third-party APIs and services | Primarily designed for graph databases and linked data | Limited integrations primarily focused on automation | Good integration with existing tools and APIs | Modular integration possibilities, but may be limited in scope |

### Conclusion:
- **LangChain** would be a strong choice if you're looking for a versatile and widely supported framework, especially if you're dealing with diverse applications.
- **LangGraph** is better suited for projects that rely heavily on graph-based reasoning.
- **AutoGen** could be beneficial for quickly prototyping AI agents, especially if you want to automate much of the generation process.
- **CrewAI** would be ideal if collaboration among multiple AI agents is a key requirement for your project.
- **Baby AGI** might be appropriate if you're looking for a framework focused on adaptive learning and reinforcement methodologies, but it may require more expertise.

The best choice ultimately depends on your specific needs, expertise, and the required collaboration level for your AI agent.